# Sprint 1 — Validação do pipeline (PCAP → KG)

Este notebook valida as saídas do Sprint 1 e produz o relatório de validação que satisfaz os *gates* descritos em [`../README.md`](../README.md).

Carrega:
- `sessions.parquet` produzido por `build_sessions.py`
- `clusters.csv` produzido por `derive_clusters.py`

E executa quatro verificações:
1. Inventário básico (contagens, distribuições)
2. JA4 — extração e distribuição
3. Sessões — duração, atividade
4. Clusters — *ground truth* sanidade

Cada bloco é independente; rode na ordem se quiser o relatório completo.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import dotenv_values

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

# Carregar DATA_ROOT do .env
env = dotenv_values('../../.env')
DATA_ROOT = Path(env.get('DATA_ROOT', '/Volumes/Untitled/kg-ddos-data'))
DATASET = os.environ.get('DATASET', 'cicids2017')

print(f'DATA_ROOT: {DATA_ROOT}')
print(f'DATASET:   {DATASET}')

SESSIONS_PATH = DATA_ROOT / 'processed' / 'sessions' / f'{DATASET}.parquet'
CLUSTERS_PATH = DATA_ROOT / 'processed' / 'clusters' / f'{DATASET}.csv'

assert SESSIONS_PATH.exists(), f'sessões não encontradas em {SESSIONS_PATH}'
assert CLUSTERS_PATH.exists(), f'clusters não encontrados em {CLUSTERS_PATH}'

print(f'\n✅ Arquivos encontrados.')

## 1. Inventário básico

In [ ]:
sessions = pd.read_parquet(SESSIONS_PATH)
clusters = pd.read_csv(CLUSTERS_PATH)

print(f'Sessões totais: {len(sessions):,}')
print(f'Clusters totais: {len(clusters):,}')
print(f'\nColunas em sessões:')
for c in sessions.columns:
    nn = sessions[c].notna().sum()
    print(f'  {c:30s}  {nn:>10,} valores não nulos')

In [ ]:
# Distribuição de rótulos (ground truth do dataset)
label_col = 'label_first' if 'label_first' in sessions.columns else 'label'
if label_col in sessions.columns:
    counts = sessions[label_col].value_counts()
    print('Distribuição de rótulos de tráfego:\n')
    print(counts.to_string())
    
    ax = counts.plot(kind='barh', color='#1F3A5F')
    ax.set_xlabel('Número de sessões')
    ax.set_title(f'Distribuição de rótulos — {DATASET}')
    plt.tight_layout()
    plt.show()
else:
    print('Coluna de rótulo não encontrada.')

## 2. JA4 — extração e distribuição

**Gate de aprovação:** pelo menos um JA4 válido extraído por classe de tráfego com TLS.

In [ ]:
if 'ja4' not in sessions.columns:
    print('❌ Coluna ja4 não encontrada — extração de JA4 falhou ou não foi executada.')
    ja4_tls_cov, n_unique_ja4 = 0.0, 0
else:
    n_with_ja4 = int(sessions['ja4'].notna().sum())
    n_unique_ja4 = int(sessions['ja4'].dropna().nunique())
    # JA4 só existe em tráfego TLS. Medir cobertura sobre o subconjunto TLS
    # (porta 443, ou com SNI/JA3) — não sobre TODAS as sessões: HTTP/DNS puro não
    # tem JA4 por natureza, então "% de todas" mede composição de tráfego, não
    # qualidade de extração.
    is_tls = (sessions['dst_port_first'] == 443)
    if 'sni' in sessions.columns: is_tls = is_tls | sessions['sni'].notna()
    if 'ja3' in sessions.columns: is_tls = is_tls | sessions['ja3'].notna()
    n_tls = int(is_tls.sum())
    n_ja4_tls = int(sessions.loc[is_tls, 'ja4'].notna().sum())
    ja4_tls_cov = n_ja4_tls / n_tls if n_tls else 0.0
    print(f'JA4 extraído: {n_with_ja4:,} ({100*n_with_ja4/len(sessions):.1f}% de TODAS as sessões)')
    print(f'  -> JA4 só se aplica a TLS. Cobertura no subconjunto TLS: '
          f'{100*ja4_tls_cov:.1f}% ({n_ja4_tls:,}/{n_tls:,})')
    print(f'JA4 únicos: {n_unique_ja4:,}')
    if n_unique_ja4 > 0:
        print('\nTop 10 JA4 mais frequentes:')
        for ja4, cnt in sessions['ja4'].value_counts().head(10).items():
            print(f'  {ja4[:50]:50s}  {cnt:>6,}')
        if label_col in sessions.columns:
            top_ja4 = sessions['ja4'].value_counts().head(15).index
            mask = sessions['ja4'].isin(top_ja4)
            ct = pd.crosstab(sessions.loc[mask, 'ja4'], sessions.loc[mask, label_col])
            fig, ax = plt.subplots(figsize=(11, 6))
            sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', ax=ax, cbar_kws={'label': 'sessões'})
            ax.set_title('JA4 (top 15) × Rótulo — concentração indica assinatura de cliente coerente')
            plt.tight_layout(); plt.show()

## 3. Sessões — duração, atividade

**Gate de aprovação:** distribuição de duração consistente com PCAPs originais (Slow HTTP attacks têm sessões longas).

In [ ]:
if 'duration_s' in sessions.columns:
    desc = sessions['duration_s'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
    print('Duração das sessões (segundos):\n')
    print(desc.to_string())
    
    # Comparar duração por rótulo
    if label_col in sessions.columns:
        fig, ax = plt.subplots(figsize=(11, 5))
        top_labels = sessions[label_col].value_counts().head(10).index
        plot_df = sessions[sessions[label_col].isin(top_labels)].copy()
        plot_df['duration_s'] = plot_df['duration_s'].clip(upper=plot_df['duration_s'].quantile(0.99))
        sns.boxplot(data=plot_df, x='duration_s', y=label_col, ax=ax, color='#1F3A5F')
        ax.set_xlabel('Duração da sessão (s, clipped a P99)')
        ax.set_title('Duração de sessão por rótulo')
        plt.tight_layout()
        plt.show()
else:
    print('Coluna duration_s não encontrada.')

In [ ]:
# Requisições por sessão
if 'n_requests' in sessions.columns:
    fig, ax = plt.subplots(figsize=(11, 4))
    sessions['n_requests'].clip(upper=sessions['n_requests'].quantile(0.99)).hist(
        bins=50, ax=ax, color='#1F3A5F', edgecolor='white'
    )
    ax.set_xlabel('Requisições por sessão (clipped a P99)')
    ax.set_ylabel('Sessões')
    ax.set_title('Distribuição de requisições por sessão')
    plt.tight_layout()
    plt.show()

## 4. Clusters — *ground truth*

**Gate de aprovação:** pelo menos 10 *clusters* manualmente validados em `clusters.csv.sample.csv`.

In [ ]:
print(f'Total de clusters: {len(clusters):,}')

if 'n_sessions' in clusters.columns:
    print(f'\nSessões por cluster:')
    print(clusters['n_sessions'].describe().to_string())
    
    fig, ax = plt.subplots(figsize=(11, 4))
    clusters['n_sessions'].clip(upper=clusters['n_sessions'].quantile(0.99)).hist(
        bins=50, ax=ax, color='#1F3A5F', edgecolor='white'
    )
    ax.set_xlabel('Sessões por cluster')
    ax.set_ylabel('Frequência')
    ax.set_title('Tamanho dos clusters')
    plt.tight_layout()
    plt.show()

if 'label' in clusters.columns:
    print(f'\nDistribuição de rótulos dos clusters:')
    print(clusters['label'].value_counts().to_string())

In [ ]:
# Amostra para revisão manual (gate de aprovação)
sample_path = CLUSTERS_PATH.parent / (CLUSTERS_PATH.stem + '.sample.csv')
if sample_path.exists():
    sample = pd.read_csv(sample_path)
    print('=' * 70)
    print('AMOSTRA DE CLUSTERS PARA REVISÃO MANUAL')
    print('=' * 70)
    print(f'\nArquivo: {sample_path}')
    print(f'Tamanho: {len(sample)} clusters\n')
    print(sample.to_string(max_cols=10))
    print('\n👀 Inspecione cada linha:')
    print('   - rótulo coerente com n_sessions?')
    print('   - duração faz sentido para o tipo de ataque?')
    print('   - cluster_start/cluster_end dentro da janela do PCAP?')
else:
    print(f'⚠️  Sample não encontrado em {sample_path}')
    print('   Rode primeiro: make clusters')

## Relatório final — gates de aprovação

Marque cada item após inspeção:

## Gates científicos — G3 (ROC AUC) e G4 (coordinatedHTTPFlood)

Computa o score de coordenação Ω(S) (sub-relações TLS/JA4 + EndpointConvergence +
NetworkProximity), o classifier de coordenação (**G3**) e a regra
`coordinatedHTTPFlood` (**G4**), reusando `scripts/compute_coordination.py`.
Clusters de detecção são **label-agnósticos** (por endpoint+janela) para evitar
circularidade; labels só avaliam.

In [ ]:
import sys
if '../scripts' not in sys.path: sys.path.append('../scripts')
import numpy as np
from compute_coordination import (assign_detection_clusters, compute_omega,
                                   session_features, gate_g3, gate_g4)

_df = assign_detection_clusters(sessions, window_s=300)
_clusters = compute_omega(_df)
_benign = _clusters[_clusters['dominant_label'] == 'BENIGN']
g4_tau = max(float(np.percentile(_benign['omega'], 99)) if len(_benign) else 1.0, 1.0)
_http = {80, 443, 8080, 8000, 8008, 8443, 8888}
g4_hits = gate_g4(_clusters, g4_tau, k_min=5, tau_rate=1.0, http_ports=_http)

_df = session_features(_df)
auc_lr, auc_rf, _feats, _imp, _abl = gate_g3(_df)
g3_auc = max(auc_lr, auc_rf)
g3_auc_coord = _abl['coordination_only']

print(f'G3 ROC AUC: melhor={g3_auc:.4f}  '
      f'(coordenação-só={g3_auc_coord:.4f}, comportamental-só={_abl["behavioral_only"]:.4f})')
print(f'   caveat: AUC alto é parte trivial (CIC é separável por fluxo); '
      f'a evidência limpa da hipótese é a ablação coordenação-só.')
print(f'\nG4 coordinatedHTTPFlood: {len(g4_hits)} clusters (τ={g4_tau:.1f}, |S|≥5, rate≥1, HTTP), '
      f'{int((g4_hits["attack_frac"]>=0.5).sum())}/{len(g4_hits)} attack-dominant')
print('\nTop 5 clusters por Ω:')
print(g4_hits.head(5)[['endpoint', 'omega', 'size', 'dominant_label']].to_string(index=False))

In [ ]:
n_sessions = len(sessions)
n_clusters = len(clusters)
sample_exists = sample_path.exists()

gates = [
    ('Sprint 1 produziu ≥ 1.000 sessões',
     n_sessions >= 1000, f'{n_sessions:,} sessões'),
    ('G1: ≥ 5 fingerprints JA4 distintos',
     n_unique_ja4 >= 5, f'{n_unique_ja4} JA4 únicos'),
    ('JA4 cobre ≥ 50% do tráfego TLS (não de todo o tráfego)',
     ja4_tls_cov >= 0.50, f'{100*ja4_tls_cov:.1f}% do TLS'),
    ('≥ 10 clusters derivados',
     n_clusters >= 10, f'{n_clusters} clusters'),
    ('Amostra de clusters para revisão existe',
     sample_exists, str(sample_path)),
    ('G3: ROC AUC ≥ 0.85 (coordenação separa ataque/benigno)',
     g3_auc >= 0.85, f'AUC={g3_auc:.4f} (coord-só={g3_auc_coord:.4f})'),
    ('G4: coordinatedHTTPFlood retorna ≥ 1 cluster',
     len(g4_hits) >= 1, f'{len(g4_hits)} clusters (τ={g4_tau:.1f})'),
]

print('=' * 70)
print('GATES DE APROVAÇÃO — Sprint 1')
print('=' * 70)
for desc, ok, val in gates:
    icon = '✅' if ok else '❌'
    print(f'{icon}  {desc:58s}  → {val}')

print()
if all(g[1] for g in gates):
    print('🎉 Todos os gates passaram. Sprint 1 validado.')
else:
    print('⚠️  Alguns gates falharam. Revise os passos antes de prosseguir.')

## Query SPARQL contra Fuseki (opcional — só após `make load-kg`)

Validação final: confirma que o grafo foi carregado e responde.

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

FUSEKI_URL = 'http://localhost:3030'
FUSEKI_DATASET = f'{DATASET}'
endpoint = f'{FUSEKI_URL}/{FUSEKI_DATASET}/sparql'

try:
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery('''
        PREFIX kg: <https://kg-ddos.example/ontology#>
        SELECT (COUNT(*) AS ?n) WHERE {
            ?s a kg:ApplicationSession
        }
    ''')
    sparql.setReturnFormat(JSON)
    result = sparql.query().convert()
    n = result['results']['bindings'][0]['n']['value']
    print(f'✅ Fuseki respondeu: {n} ApplicationSession no grafo')
except Exception as e:
    print(f'⚠️  Fuseki não respondeu: {e}')
    print(f'    Verifique: docker compose ps  (de experiments/sprint-1)')
    print(f'    Ou rode: make load-kg')